# Oracle 1T Option Retrieval, Ranking, and Optopsy Backtest

This notebook tests option selection inside oracle equity trade windows. The equity trade labels define entry/exit windows; option retrieval selects viable contracts at entry and prices them at the equity exit date, falling back to expiration when the option expires first.

The important separation is:

- Quant Warehouse provides source data and oracle trade labels.
- Quant Orchestrator builds experiment artifacts and trains the option selector.
- Optopsy, a third-party options backtesting engine, simulates the portfolio execution layer.

Selected-contract return summaries are diagnostics only. Portfolio-level backtest metrics come from Optopsy.

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name != 'quant-orchestrator':
    REPO_ROOT = next(parent for parent in Path.cwd().resolve().parents if parent.name == 'quant-orchestrator')
WAREHOUSE_ROOT = REPO_ROOT.parent / 'quant-warehouse'
for path in (REPO_ROOT, WAREHOUSE_ROOT):
    resolved = str(path.resolve())
    if resolved in sys.path:
        sys.path.remove(resolved)
    sys.path.insert(0, resolved)
for module_name in list(sys.modules):
    if module_name == 'quant_orchestrator' or module_name.startswith('quant_orchestrator.'):
        del sys.modules[module_name]

import pandas as pd

from quant_orchestrator.research_tools import (
    OptopsyExecutionConfig,
    OptionRetrievalConfig,
    OracleOptionExperimentConfig,
    SharedSplitConfig,
    run_oracle_option_experiment,
)

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 220)

In [2]:
CONFIG = OracleOptionExperimentConfig(
    experiment_name='oracle_1t_option_retrieval_ranking',
    symbols=('AAPL', 'AMZN', 'AVGO', 'GOOG', 'GOOGL', 'META', 'MSFT', 'NVDA', 'TSLA'),
    price_start='2018-01-01',
    price_end='2026-06-24',
    k_params={'YE': [1]},
    min_profit_pct=0.01,
    split=SharedSplitConfig(insample_end='2024-12-31'),
    retrieval=OptionRetrievalConfig(
        min_dte=30,
        max_dte=90,
        target_dte=45,
        max_abs_moneyness=0.15,
        min_entry_mid=0.25,
        max_entry_spread_pct=0.20,
        max_candidates_per_trade=40,
        exit_lookback_days=7,
    ),
    execution=OptopsyExecutionConfig(
        capital=100_000.0,
        quantity=1,
        max_positions=5,
        multiplier=100,
        selector='first',
    ),
    artifact_dir=str(REPO_ROOT / 'artifacts/options/oracle_1t_option_retrieval_ranking/latest'),
    log_mlflow=True,
)
CONFIG

OracleOptionExperimentConfig(experiment_name='oracle_1t_option_retrieval_ranking', symbols=('AAPL', 'AMZN', 'AVGO', 'GOOG', 'GOOGL', 'META', 'MSFT', 'NVDA', 'TSLA'), price_start='2018-01-01', price_end='2026-06-24', k_params={'YE': [1]}, min_profit_pct=0.01, split=SharedSplitConfig(insample_end='2024-12-31'), retrieval=OptionRetrievalConfig(min_dte=30, max_dte=90, target_dte=45, max_abs_moneyness=0.15, min_entry_mid=0.25, max_entry_spread_pct=0.2, max_candidates_per_trade=40, exit_lookback_days=7, chain_columns=('snapshot_date', 'contract_symbol', 'expiration', 'strike', 'option_type', 'bid', 'ask', 'mid', 'delta', 'gamma', 'theta', 'vega', 'rho', 'iv', 'implied_volatility', 'volume', 'open_interest')), execution=OptopsyExecutionConfig(capital=100000.0, quantity=1, max_positions=5, multiplier=100, selector='first'), random_seed=20260702, log_mlflow=True, mlflow_experiment='options_trading', mlflow_tracking_uri=None, artifact_dir='/home/jlee153232/PycharmProjects/quant-orchestrator/arti

In [3]:
result = run_oracle_option_experiment(CONFIG)
print(f'elapsed_seconds={result.elapsed_seconds:.2f}')
print(f'mlflow_run_id={result.mlflow_run_id}')
print('artifacts:')
for name, path in result.artifact_paths.items():
    print(f'  {name}: {path}')

/home/jlee153232/miniconda3/lib/python3.13/site-packages/empyrical/stats.py:1424: RuntimeWarning: invalid value encountered in scalar divide
  return np.abs(np.percentile(returns, 95)) / np.abs(np.percentile(returns, 5))
/home/jlee153232/miniconda3/lib/python3.13/site-packages/empyrical/stats.py:1424: RuntimeWarning: invalid value encountered in scalar divide
  return np.abs(np.percentile(returns, 95)) / np.abs(np.percentile(returns, 5))
/home/jlee153232/miniconda3/lib/python3.13/site-packages/empyrical/stats.py:1424: RuntimeWarning: invalid value encountered in scalar divide
  return np.abs(np.percentile(returns, 95)) / np.abs(np.percentile(returns, 5))


elapsed_seconds=50.77
mlflow_run_id=adf2c89da5a9448385eb913253df7442
artifacts:
  coverage: /home/jlee153232/PycharmProjects/quant-orchestrator/artifacts/options/oracle_1t_option_retrieval_ranking/latest/coverage.parquet
  oracle_trades: /home/jlee153232/PycharmProjects/quant-orchestrator/artifacts/options/oracle_1t_option_retrieval_ranking/latest/oracle_trades.parquet
  option_panel: /home/jlee153232/PycharmProjects/quant-orchestrator/artifacts/options/oracle_1t_option_retrieval_ranking/latest/option_candidate_panel.parquet
  train_panel: /home/jlee153232/PycharmProjects/quant-orchestrator/artifacts/options/oracle_1t_option_retrieval_ranking/latest/train_panel.parquet
  eval_panel: /home/jlee153232/PycharmProjects/quant-orchestrator/artifacts/options/oracle_1t_option_retrieval_ranking/latest/eval_panel.parquet
  selector_summary: /home/jlee153232/PycharmProjects/quant-orchestrator/artifacts/options/oracle_1t_option_retrieval_ranking/latest/selector_summary.csv
  optopsy_summary: /home

## Coverage and Panels

In [4]:
display(result.coverage)
print('oracle_trades', len(result.oracle_trades))
print('option_panel_rows', len(result.option_panel), 'trade_list', result.option_panel['trade_id'].nunique() if not result.option_panel.empty else 0)
print('train_rows', len(result.train_panel), 'eval_rows', len(result.eval_panel))

,symbol,row_count,snapshot_day_count,min_snapshot_date,max_snapshot_date
0,NVDA,709954,2111,2018-01-02,2026-06-23
1,TSLA,693226,2114,2018-01-02,2026-06-24
2,GOOGL,691906,2110,2018-01-02,2026-06-24
3,AMZN,686606,2113,2018-01-02,2026-06-24
4,GOOG,670120,2110,2018-01-02,2026-06-24
5,AAPL,665512,2113,2018-01-02,2026-06-24
6,MSFT,645438,2113,2018-01-02,2026-06-24
7,AVGO,636940,2113,2018-01-02,2026-06-24
8,META,338205,1156,2021-07-08,2026-06-24


oracle_trades 162
option_panel_rows 2400 trade_list 73
train_rows 1267 eval_rows 1133


## Selector Diagnostics

These rows summarize selected option contracts before portfolio execution constraints. They are useful for ranking quality, but they are not the portfolio backtest.

In [5]:
display(result.selector_summary)
display(result.symbol_summary.head(30))

,selector,trades,symbols,mean_return,median_return,win_rate,avg_spread_pct,avg_abs_moneyness,avg_dte
0,model_ranker,34,9,4.840456,3.097718,0.941176,0.064729,0.077026,38.323529
1,oracle_best_possible,34,9,8.357153,5.793905,1.000000,0.070734,0.089969,46.176471
2,fixed_near_atm,34,9,2.987513,2.821944,0.941176,0.035756,0.012136,42.294118


,symbol,side,trades,mean_return,win_rate
8,GOOGL,long,2,13.137904,1.0
4,AVGO,long,2,9.731929,1.0
6,GOOG,long,2,7.922567,1.0
3,AMZN,short,2,7.386487,1.0
5,AVGO,short,1,7.281853,1.0
14,NVDA,long,2,6.734374,1.0
12,MSFT,long,2,6.112857,1.0
10,META,long,1,5.459494,1.0
9,GOOGL,short,2,5.221316,1.0
13,MSFT,short,2,5.092411,1.0


## Optopsy Portfolio Backtest

These are the actual options backtest summaries. Optopsy applies capital, contract quantity, multiplier, overlap, and max-position constraints.

In [6]:
display(result.optopsy_summary)

,selector,framework,selected_rows,closed_trades,equity_points,final_equity,total_trades,winning_trades,losing_trades,win_rate,total_pnl,total_return,avg_pnl,avg_win,avg_loss,max_win,max_loss,profit_factor,max_drawdown,avg_days_in_trade,sharpe_ratio,sortino_ratio,var_95,cvar_95,calmar_ratio,omega_ratio,tail_ratio
0,model_ranker,optopsy,34,19,19,176117.5,19,18,1,0.947368,76117.5,0.761175,4006.184211,4229.583333,-15.0,20514.5,-15.0,5075.500000,-0.000103,40.842105,1.511092,6.472319,0.216079,-0.022814,3.792702,6.071458,0.0
1,oracle_best_possible,optopsy,34,20,20,184581.5,20,20,0,1.000000,84581.5,0.845815,4229.075000,4229.075000,0.0,17734.5,857.5,inf,0.000000,45.950000,1.216547,2.643911,1.299426,0.799534,2.314860,3.466372,0.0
2,fixed_near_atm,optopsy,34,19,19,186513.5,19,17,2,0.894737,86513.5,0.865135,4553.342105,5147.205882,-494.5,14458.5,-861.5,88.475733,-0.005064,42.842105,2.002777,10.286277,-0.231205,-0.993084,8.411974,12.708951,0.0


## Written Analysis

In [7]:
from IPython.display import Markdown
Markdown(result.analysis_markdown)

## Written Analysis

- Oracle equity trades generated: 162.
- Shared split: in-sample <= 2024-12-31, out-of-sample >= 2025-01-01.
- Option retrieval produced 2,400 contract rows across 73 oracle trade windows.
- Train rows: 1,267; eval rows: 1,133.
- Option ranker metrics: train_r2=0.780, eval_r2=-0.255, eval_mae=2.379.
- ThetaData option feature columns available in this run: liquidity_score.
- Greeks/IV missing from the cached chains for this run: delta, gamma, theta, vega, rho, iv. The feature-engineering API will include them automatically when present in ThetaData storage.
- model_ranker: 34 eval trades, mean=484.05%, median=309.77%, win_rate=94.12%.
- oracle_best_possible: 34 eval trades, mean=835.72%, median=579.39%, win_rate=100.00%.
- fixed_near_atm: 34 eval trades, mean=298.75%, median=282.19%, win_rate=94.12%.
- Portfolio execution is simulated by the third-party Optopsy engine; Quant Orchestrator only adapts selected contracts into Optopsy's raw-trade schema.
- Optopsy model_ranker: closed_trades=19, final_equity=176,117.50, total_return=76.12%.
- Optopsy oracle_best_possible: closed_trades=20, final_equity=184,581.50, total_return=84.58%.
- Optopsy fixed_near_atm: closed_trades=19, final_equity=186,513.50, total_return=86.51%.
- These artifacts are experiment products, not permanent Quant Warehouse source-of-truth tables.